In [ ]:
from pathlib import Path
import re
import pandas as pd
from IPython.display import display, Markdown

from pexl.validate.schema import (
    diff_schema_dirs,
    schema_diff_to_markdown,
    SchemaDiffOptions,
)

# Resolve repository root
ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "schemas").is_dir():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Cannot find data/schemas")
    ROOT = ROOT.parent

SCHEMA_DIR = ROOT / "data" / "schemas"
OUTPUT_DIR = ROOT / "data" / "exports" / "schema_evolution"

INCLUDE_DEV = False
INCLUDE_CURRENT = False

def version_key(path):
    name = path.name.lower()
    if name == "current":
        return (float("inf"),)
    numbers = tuple(map(int, re.findall(r"\d+", name)))
    return numbers or (float("inf"),)

schema_dirs = [
    p for p in SCHEMA_DIR.iterdir()
    if p.is_dir()
    and (INCLUDE_CURRENT or p.name != "current")
    and (p.name == "current" or re.match(r"^v?\d", p.name))
]

schema_dirs = sorted(
    schema_dirs,
    key=lambda p: (p.name == "current", version_key(p))
)

print(f"Found {len(schema_dirs)} schemas:")
for p in schema_dirs:
    print("  ", p.name)

Found 11 schemas:
   v1.06
   v1.07
   v1.08
   v1.09
   v1.10_4
   v1_11_4
   v1_11_5
   v1_11_7
   v1_12_7
   1.13.1_dev
   current


In [5]:
# Large cap to avoid truncating changes in historical analysis
options = SchemaDiffOptions(max_value_changes=100_000)

diff_chain = []

for old_dir, new_dir in zip(schema_dirs[:-1], schema_dirs[1:]):
    print(f"{old_dir.name} → {new_dir.name}")

    report = diff_schema_dirs(
        old_dir,
        new_dir,
        options=options,
    )

    diff_chain.append({
        "old": old_dir.name,
        "new": new_dir.name,
        "report": report,
    })

print(f"\nGenerated {len(diff_chain)} diffs.")

v1.06 → v1.07
v1.07 → v1.08
v1.08 → v1.09
v1.09 → v1.10_4
v1.10_4 → v1_11_4
v1_11_4 → v1_11_5
v1_11_5 → v1_11_7
v1_11_7 → v1_12_7
v1_12_7 → 1.13.1_dev
1.13.1_dev → current

Generated 10 diffs.


In [6]:
variable_changes = []
value_changes = []
summary = []

for entry in diff_chain:
    old, new = entry["old"], entry["new"]
    report = entry["report"]

    for table_name, table in report["tables"].items():
        rows = table.get("rows", {})
        changes = table.get("changes", {})

        # Added / removed variables
        for kind, key in [
            ("added", "var_name_added"),
            ("removed", "var_name_removed"),
        ]:
            for var in rows.get(key, []):
                variable_changes.append({
                    "from": old,
                    "to": new,
                    "table": table_name,
                    "variable": var,
                    "change": kind,
                })

        # Modified metadata / formulas / labels
        for change in changes.get("value_changes", []):
            value_changes.append({
                "from": old,
                "to": new,
                "table": table_name,
                "variable": change.get("var_name"),
                "column": change.get("column"),
                "old_value": change.get("old"),
                "new_value": change.get("new"),
            })

        summary.append({
            "from": old,
            "to": new,
            "table": table_name,
            "added": len(rows.get("var_name_added", [])),
            "removed": len(rows.get("var_name_removed", [])),
            "modified": changes.get("value_change_count", 0),
            "truncated": changes.get("truncated", False),
        })

df_variables = pd.DataFrame(
    variable_changes,
    columns=["from", "to", "table", "variable", "change"],
)
df_values = pd.DataFrame(
    value_changes,
    columns=[
        "from", "to", "table", "variable",
        "column", "old_value", "new_value",
    ],
)
df_summary = pd.DataFrame(summary)

display(df_summary)

if df_summary["truncated"].any():
    print("WARNING: Some value changes were truncated.")

,from,to,table,added,removed,modified,truncated
0,v1.06,v1.07,IN,0,1,0,False
1,v1.06,v1.07,OUT,0,1,0,False
2,v1.06,v1.07,PV-Import,0,0,0,False
3,v1.06,v1.07,Wetter,0,0,0,False
4,v1.07,v1.08,IN,6,0,1146,False
5,v1.07,v1.08,OUT,0,0,0,False
6,v1.07,v1.08,SIM,0,0,0,False
7,v1.08,v1.09,IN,1,1,92,False
8,v1.08,v1.09,OUT,0,0,11,False
9,v1.08,v1.09,SIM,0,0,3,False


In [7]:
# Adjust or extend search terms as needed
PATTERNS = {
    "GWP": r"gwp|co2|emission|carbon",
    "Primary Energy": r"primary|primär|^pe_|pec_|fpea|dxe",
    "PV": r"pv_|photovolta|roof",
    "Version / Date": r"version|datum|date",
}

def filter_changes(pattern):
    vars_match = df_variables[
        df_variables["variable"].str.contains(
            pattern, case=False, regex=True, na=False
        )
    ]

    values_match = df_values[
        df_values["variable"].str.contains(
            pattern, case=False, regex=True, na=False
        )
    ]

    return vars_match, values_match


for title, pattern in PATTERNS.items():
    added_removed, modified = filter_changes(pattern)

    display(Markdown(f"### {title}"))

    display(Markdown("**Added / Removed**"))
    display(added_removed)

    display(Markdown("**Modified properties**"))
    display(modified)

### GWP

**Added / Removed**

,from,to,table,variable,change
2,v1.07,v1.08,IN,GWP_direct_biogenic,added
3,v1.07,v1.08,IN,GWP_direct_biogenic_share,added
4,v1.07,v1.08,IN,GWP_direct_fossile,added
5,v1.07,v1.08,IN,GWP_direct_life,added
66,v1.09,v1.10_4,IN,GWP_ceilings_name,removed
...,...,...,...,...,...
1005,1.13.1_dev,current,OUT,legacy_GWP_life_tga_ventilation,removed
1006,1.13.1_dev,current,OUT,legacy_GWP_life_walls,removed
1007,1.13.1_dev,current,OUT,legacy_GWP_life_windows,removed
1009,1.13.1_dev,current,OUT,test_GWP_sum,removed


**Modified properties**

,from,to,table,variable,column,old_value,new_value
262,v1.07,v1.08,IN,GWP_battery_name,ka,2.0,2
263,v1.07,v1.08,IN,GWP_battery_name,Type,nan,userinput
264,v1.07,v1.08,IN,GWP_boreholes_name,ka,2.0,2
265,v1.07,v1.08,IN,GWP_boreholes_name,Type,nan,userinput
266,v1.07,v1.08,IN,GWP_ceilings_name,ka,2.0,2
...,...,...,...,...,...,...,...
2743,1.13.1_dev,current,IN,GWP_roof_name,Formel,☁THG'!D23,THG!F17
2744,1.13.1_dev,current,IN,GWP_walls_name,ka,nan,2.0
2745,1.13.1_dev,current,IN,GWP_walls_name,Formel,☁THG'!D21,THG!F15
2746,1.13.1_dev,current,IN,GWP_windows_name,ka,nan,2.0


### Primary Energy

... (10 outputs left out)

**Added / Removed**

,from,to,table,variable,change
7,v1.07,v1.08,IN,scenario_version,added
8,v1.08,v1.09,IN,input_version_number,added
57,v1.09,v1.10_4,IN,input_version_date,added
79,v1.09,v1.10_4,IN,ghg_conversion_factor_gasoline,removed
102,v1.09,v1.10_4,OUT,project_creation_date,added
144,v1.10_4,v1_11_4,OUT,project_creation_date,removed
232,v1_11_7,v1_12_7,IN,ghg_conversion_factor_gasoline,added
258,v1_11_7,v1_12_7,IN,tool_version,added
261,v1_11_7,v1_12_7,IN,input_version_date,removed
267,v1_11_7,v1_12_7,IN,scenario_version,removed


**Modified properties**

,from,to,table,variable,column,old_value,new_value
90,v1.07,v1.08,IN,DHW_conversion_1,ka,0.0,0
91,v1.07,v1.08,IN,DHW_conversion_1,Type,Zwischenrechnung,calculation
92,v1.07,v1.08,IN,DHW_conversion_2,ka,0.0,0
93,v1.07,v1.08,IN,DHW_conversion_2,Type,Zwischenrechnung,calculation
666,v1.07,v1.08,IN,district_heating_conversion_profile,ka,1.0,1
667,v1.07,v1.08,IN,district_heating_conversion_profile,Type,nan,value
710,v1.07,v1.08,IN,ghg_conversion_factor_gasoline,ka,0.0,0
711,v1.07,v1.08,IN,ghg_conversion_factor_gasoline,Type,nan,obsolet
892,v1.07,v1.08,IN,pe_conversion_factor_gasoline,ka,1.0,1
893,v1.07,v1.08,IN,pe_conversion_factor_gasoline,Type,nan,value


In [8]:
oldest = schema_dirs[0]
newest = schema_dirs[-1]

full_diff = diff_schema_dirs(
    oldest,
    newest,
    options=options,
)

print(f"Full comparison: {oldest.name} → {newest.name}")

for table_name, table in full_diff["tables"].items():
    rows = table.get("rows", {})

    print(f"\n{table_name}")
    print("  Added:", len(rows.get("var_name_added", [])))
    print("  Removed:", len(rows.get("var_name_removed", [])))
    print(
        "  Modified:",
        table.get("changes", {}).get("value_change_count", 0),
    )

Full comparison: v1.06 → current

IN
  Added: 67
  Removed: 24
  Modified: 73

OUT
  Added: 23
  Removed: 5
  Modified: 0

SIM
  Added: 0
  Removed: 0
  Modified: 0

PV-Import
  Added: 0
  Removed: 0
  Modified: 0

Wetter
  Added: 0
  Removed: 0
  Modified: 0


In [9]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Individual Markdown diffs
for entry in diff_chain:
    md = schema_diff_to_markdown(
        entry["report"],
        value_format="items",
    )

    filename = f"{entry['old']}__{entry['new']}.md"
    (OUTPUT_DIR / filename).write_text(
        md, encoding="utf-8"
    )

# Full comparison
full_md = schema_diff_to_markdown(
    full_diff,
    value_format="items",
)

(OUTPUT_DIR / "full_comparison.md").write_text(
    full_md, encoding="utf-8"
)

# Consolidated change history
df_variables.to_csv(
    OUTPUT_DIR / "variable_changes.csv",
    index=False,
)
df_values.to_csv(
    OUTPUT_DIR / "value_changes.csv",
    index=False,
)
df_summary.to_csv(
    OUTPUT_DIR / "summary.csv",
    index=False,
)

print(f"Exported to: {OUTPUT_DIR}")

Exported to: C:\Users\schneids\code\peexcel\data\exports\schema_evolution
